# 2. Evaluating the environmental impacts associated with meeting Decent Living Standards under French neutrality scenarios Transition(s) 2050

In [ ]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc
from premise import *
from datapackage import Package
import pandas as pd
import numpy as np

# `🔧` Setup

In [ ]:
bd.projects

In [ ]:
NAME_BW_PROJECT="ADEME_DLS_10_scenarios"

In [ ]:
#Set in the right project and print the databases
bd.projects.set_current(NAME_BW_PROJECT)
bd.databases

**After downloading the [PB-LCIA package](https://github.com/gpuigsamper/PB-LCIA/tree/ei_310):**

In [ ]:
pip install "PATHWAY" --force-reinstall

In [ ]:
import aesa_pbs
aesa_pbs.add_aesa_pbs()

# `⛰️` Impact assessment methods

In [ ]:
#PB-LCIA

pb_lcia = [[me for me in bd.methods if 'AESA' in str(me) and 'freshwater' in str(me)][0],
      [me for me in bd.methods if 'AESA' in str(me) and 'ozone' in str(me)][0],
      [me for me in bd.methods if 'AESA' in str(me) and 'nitrogen' in str(me) and 'inverse modelling, surface water' in str(me)][0],
      [me for me in bd.methods if 'AESA' in str(me) and 'atmospheric aerosol loading' in str(me)][0],
      [me for me in bd.methods if 'AESA' in str(me) and 'ocean acidification' in str(me)][0],
      [me for me in bd.methods if 'AESA' in str(me) and 'change in biosphere integrity' in str(me) and 'total' in str(me)][0],
      [me for me in bd.methods if 'AESA' in str(me) and 'phosphorus' in str(me)][0],
      [me for me in bd.methods if 'AESA' in str(me) and 'climate change' in str(me)][0],
      [me for me in bd.methods if 'AESA' in str(me) and 'climate change' in str(me)][1],
      [me for me in bd.methods if 'AESA' in str(me) and 'land-system change' in str(me)][0]
     ]

#IPCC 2021
# Approach 0/0
IPCC_21 = [m for m in bd.methods if 'IPCC 2021' in str(m) 
            and 'global warming potential (GWP100)' in str(m) 
            and not 'fossil' in str(m) 
            and not 'biogenic' in str(m) 
            and not 'land use' in str(m) 
            and not 'SLCFs' in str(m)
            and not 'LT' in str(m)]

impact_cats = IPCC_21 + pb_lcia

# `🔧` Manipulating multiple databases

In [ ]:
#generate a list of names of generated databases by premise
premise_db_name_list=[]
for db_name in bd.databases.keys():
    if "ei_cutoff" in db_name:
        premise_db_name_list.append(db_name)
premise_db_name_list

In [ ]:
#generate a list of generated databases by premise
premise_db_list=[]
for db_name in premise_db_name_list:
    premise_db_list.append(bd.Database(db_name))

In [ ]:
#Options for model / SSP / IAM / FR scenarios
model_list=['image','tiam-ucl','remind', 'none']
year_list=['2020','2030','2050']
SSP_list=['SSP1','SSP2','SSP3','SSP4','SSP5']
IAM_scenario_list=['Base','RCP26','RCP45','Npi','RCP19','H','M','VLHO','none']
FR_scenario_list=['S1','S2','S3 Nuc','S3 Renew','S4']

In [ ]:
#tag the database with corresponding year, model, IAM scenario and FR scenario
for db in premise_db_list:
    for year in year_list:
        if year in db.name:
            db.year=int(year)
    for model in model_list:
        if model in db.name:
            db.model=model
    for IAM_scenario in IAM_scenario_list:
        if IAM_scenario in db.name:
            db.IAM_scenario=IAM_scenario      
    for FR_scenario in FR_scenario_list:
        if FR_scenario in db.name:
            db.FR_scenario=FR_scenario    
    #Warning
    db.warning=''

In [ ]:
#If you want to run the tests on all premise databases
selected_db_list=[a for a in premise_db_list if "ei_cutoff_3.10.1_image" in a.name]
selected_db_list

# `🌎` Quantification of impacts

In [ ]:
#For Decent Living Standards activities
act_name_list = [act["name"] for act in selected_db_list[0] if "DLS" in act["name"]]
act_name_list

In [ ]:
df=pd.DataFrame([],columns=['db_name','model','IAM scenario','year','warning','act','impact_category','impact','unit'])

for db in selected_db_list:
    print(db)    
    for act_name in act_name_list:
        act = [i for i in db if act_name==i["name"] and i["location"]=="FR"][0]
        print(act)
        for cat in impact_cats:
            fu = 1 
            lca = act.lca(method=cat, amount=fu)
            score = lca.score
            unit_impact = bd.Method(cat).metadata["unit"]
            label = ",".join(cat[2:4])
            df.loc[len(df.index)] = [db.name, db.model, db.IAM_scenario, db.year, db.warning, act["name"], label, score, unit_impact]

In [ ]:
#total impact for providing DLS per capita per year
#df_DLS: global impact per category and scenario
#df_DLS_act: global impact per category and scenario, broken down by dimension
db_list = list(df["db_name"].unique())

df_DLS=pd.DataFrame([],columns=['db_name', 'IAM scenario', 'year', 'activity', 'impact category', 'value', "unit"])
df_DLS_dimension=pd.DataFrame([],columns=['db_name', 'IAM scenario', 'year', 'activity', 'impact category', 'value', "share", "unit"])
for db in selected_db_list:
    for cat in impact_cats:
        label = ",".join(cat[2:4])           
        db_rows = df[(df["db_name"] == db.name)&(df["impact_category"] == label)]
        #Impact per scenario/impact category
        total_impact = db_rows["impact"].sum()
        unit_impact = bd.Method(cat).metadata["unit"]
        df_DLS.loc[len(df_DLS.index)] = [db.name, db.IAM_scenario, db.year, "DLS total", label, total_impact, unit_impact]
        for act in df["act"].unique():
            db_rows_act = df[(df["db_name"] == db.name)&(df["impact_category"] == label)&(df["act"] == act)]
            #Impact per dimension/scenario/impact category
            total_impact_dimension = db_rows_act["impact"].sum()
            #Contribution per dimension
            share = total_impact_dimension / total_impact
            df_DLS_dimension.loc[len(df_DLS_dimension.index)] = [db.name, db.IAM_scenario, db.year, act, label, total_impact_dimension, share, unit_impact]

In [ ]:
# IMAGE model assumptions for world population in 2020, 2030 and 2050
pop_df = pd.DataFrame({"Year": [2020, 2030, 2050],
                       "Population FR": [67287241,68553816,69206324],
                       "Population_world_M": [7855075060,8536033203,9646216797],
                       "Population_world_VLHO": [7855075060,8536033203,9646216797],
                       "Population_world_H": [7855075060,8580915000,10116440000]
                       }
                )

#Impacts at the French national level considering population projections

df_DLS_FR =df_DLS.copy()
for year in pop_df.index:
        df_DLS_FR.loc[df_DLS["year"]==year, "value"] = df_DLS.loc[df_DLS["year"]==year, "value"] * pop_df.loc[year, "Population FR"]

# `🍲` Introduction of a new diet

In [ ]:
#Import the new diet from the Excel file. Source: Schleiser et al. (2024)
new_diet = bi.ExcelImporter("Willet_diets_bw2.xlsx")

In [ ]:
new_diet.apply_strategies()

In [ ]:
# The diet is introduced into the M scenario database by 2050 for illustration
db_scenario_match_name = "DB_NAME" # Replace with the actual database name you want to match against
new_diet.match_database(fields=["name", "code", "unit", "location"])
new_diet.match_database(fields=["name", "unit", "location"],
                          db_name=db_scenario_match_name)

In [ ]:
new_diet.statistics()

In [ ]:
[a for a in new_diet.unlinked]

In [ ]:
new_diet.write_database(overwrite=True)

In [ ]:
#Introducing the new diet into the scenario database
db_Tr_scenario = bd.Database("DB_NAME")
db_willet_diet = bd.Database("Willet_diets_bw2")

cooking_acts =db_Tr_scenario.search("DLS cooking")
Tr_diet = db_Tr_scenario.search("food provisioning")[0]
willet_diet = db_willet_diet.search("food provisioning")[0]

In [ ]:
#adding the new diet to the cooking activities
for act in cooking_acts:
    act.new_exchange(
        input=willet_diet,
        amount=1.0,
        type="technosphere"
            ).save()

In [ ]:
#removing the old diet from the cooking activities
for act in cooking_acts:
    for exc in act.exchanges():
        if exc.input == Tr_diet:
            exc.delete()

In [ ]:
#Quantifying the impact of the new diet on the environmental impacts of DLS in France (per capita)
df_new_diet=pd.DataFrame([],columns=['db_name','model','IAM scenario','year','warning','act','impact_category','impact','unit'])
    
for act_name in act_name_list:
    act = [i for i in db_Tr_scenario if act_name==i["name"] and i["location"]=="FR"][0]
    print(act)
    for cat in impact_cats:
        fu = 1 
        lca = act.lca(method=cat, amount=fu)
        score = lca.score
        unit_impact = bd.Method(cat).metadata["unit"]
        label = ",".join(cat[2:4])
        df_new_diet.loc[len(df_new_diet.index)] = [db.name, db.model, db.IAM_scenario, db.year, db.warning, act["name"], label, score, unit_impact]

In [ ]:
df_new_diet